# PyTorch 网络层详解 - 有参数层

本 Notebook 是系列教程第2部分，**Markdown 承担全部理论讲解，代码仅做运行验证**。

## 教程覆盖模块
- 卷积层：标准Conv2d、深度可分离卷积、分组卷积、空洞卷积、转置卷积、PixelShuffle、Unfold、边界填充
- 线性层：Linear全连接、Bilinear双线性融合
- 归一化层：BN/LN/IN/GN/SyncBN，谱归一化、权重归一化
- 嵌入层：Embedding词嵌入、EmbeddingBag聚合嵌入
- 循环序列层：RNN/LSTM/GRU、单步Cell、变长PackedSequence
- Transformer组件：多头注意力、编码器、解码器、完整Transformer
- 权重初始化通用方案

## 阅读规范
1. 上方Markdown：公式、原理、适用场景、**参数对照表（含官方默认值）**、选型指南，每层单独完整说明
2. 下方Code：可直接运行的验证代码，全部顶格靠左，带详细中文注释，无文字讲解输出
3. 代码输出仅用于维度、参数量、数值校验

# 前置环境导入

加载PyTorch核心库、序列处理工具、数值辅助库；固定全局随机种子保证实验可复现；屏蔽警告并打印环境校验信息。

In [ ]:
# ====================== 全局环境导入 ======================
# 导入PyTorch核心库
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# 导入序列处理相关工具（变长序列打包/解包）
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
# 导入NumPy用于辅助数值计算
import numpy as np
# 导入warnings用于屏蔽运行时警告信息
import warnings
warnings.filterwarnings('ignore')
# 固定随机种子，确保每次运行结果可复现
torch.manual_seed(42)
np.random.seed(42)
# 打印PyTorch版本和CUDA可用状态，校验运行环境
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# 一、卷积层 总述

卷积层是CNN的基础有参数层，核心作用是**滑动窗口提取图像局部空间特征**。
内部可学习参数：卷积核权重 + 偏置bias；输入格式固定 `[batch, in_channels, H, W]`。
### 通用输出尺寸计算公式
$$
\text{out} = \left\lfloor \frac{\text{in} + 2\times\text{padding} - \text{dilation}\times(\text{kernel} - 1) - 1}{\text{stride}} \right\rfloor + 1
$$
### 通用参数量计算公式
$$\text{params} = \frac{in_{channels}}{groups} \times out_{channels} \times k^2 + out_{channels} \times bias$$

In [ ]:
# ====================== 卷积尺寸/参数量计算工具函数 ======================
# 功能：根据输入尺寸和卷积参数，计算输出特征图尺寸
# 参数：in_size-输入尺寸, kernel_size-卷积核大小, stride-步长, padding-填充, dilation-空洞率
# 返回：输出尺寸（整数）
def conv_output_size(in_size, kernel_size, stride=1, padding=0, dilation=1):
    return (in_size + 2*padding - dilation*(kernel_size-1) - 1) // stride + 1

# 功能：计算卷积层的参数量
# 参数：in_c-输入通道, out_c-输出通道, k-卷积核大小, groups-分组数, bias-是否含偏置
# 返回：总参数量（整数）
def conv_params(in_c, out_c, k, groups=1, bias=True):
    weight_params = (in_c // groups) * out_c * k * k
    bias_params = out_c if bias else 0
    return weight_params + bias_params

# 校验卷积尺寸计算公式
print("卷积尺寸校验样例:")
print(f"  input=32, kernel=3, stride=1, padding=1 -> {conv_output_size(32, 3, 1, 1)}")
print(f"  input=32, kernel=3, stride=2, padding=0 -> {conv_output_size(32, 3, 2, 0)}")
print(f"  input=32, kernel=5, stride=1, padding=2 -> {conv_output_size(32, 5, 1, 2)}")

# 校验卷积参数量计算公式，对比不同卷积变体的参数量
print("\n卷积参数量校验样例:")
print(f"标准卷积 3->16, k=3: {conv_params(3,16,3):,}")
print(f"深度可分离卷积(3->3 dw +3->16 pw): {conv_params(3,3,3)+conv_params(3,16,1):,}")
print(f"分组卷积 groups=3: {conv_params(3,16,3,groups=3):,}")

## 1.1 nn.Conv2d 标准2D卷积

### 层定义
最基础、使用最广泛的卷积层，所有输入通道与全部输出通道做全连接卷积融合。
### 数学运算
卷积核在特征图上按stride滑动，窗口内像素与卷积核权重做点积，叠加偏置得到输出单像素值。
### 核心入参（含官方默认值）
- in_channels：输入特征通道数（RGB图像为3）【必填，无默认】
- out_channels：输出特征通道数（滤波器数量）【必填，无默认】
- kernel_size：卷积核窗口尺寸【必填，无默认】
- stride：滑动步长，控制下采样，默认=1
- padding：四周补0，控制输出尺寸不变，默认=0
- dilation：空洞扩张系数，默认=1
- groups：分组数，默认=1（标准卷积）
- bias：是否启用可学习偏置，默认=True
- padding_mode：填充模式，可选zeros/reflect/replicate/circular，默认='zeros'

注：dilation参见https://blog.csdn.net/2301_77637149/article/details/142850264
- dilation=1：采样点紧密相邻，3×3 区域填满
- dilation=2：采样点横向 / 纵向各空 1 格，整体覆盖 5 格宽度
- dilation=3：采样点横向 / 纵向各空 2 格，整体覆盖 7 格宽度
### 优缺点
✅ 特征融合能力强，表达能力上限高
❌ 参数量、计算量大
### 适用场景
图像分类、目标检测、分割主干网络（ResNet、VGG等）

In [ ]:
# ====================== 1.1 标准卷积 Conv2d 验证 ======================
# 创建标准2D卷积层：输入3通道(RGB)，输出16通道，3x3卷积核，padding=1保持尺寸不变
std_conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
# 生成随机输入张量：batch=1, 通道=3, 高=32, 宽=32
x = torch.randn(1, 3, 32, 32)
# 前向传播计算输出
out = std_conv(x)
# 打印输入输出维度变化
print(f"Conv2d shape: {x.shape} -> {out.shape}")
# 计算并打印该层总参数量（权重+偏置）
total_params = sum(p.numel() for p in std_conv.parameters())
print(f"Conv2d total params: {total_params:,}")

## 1.2 逐深度卷积 DepthwiseSeparableConv

在CNN中，特征图的"深度"就是指通道数（Channel）， "逐个深度"，即一个深度（通道）一个深度（通道）地处理。对每个输入通道，单独使用一个卷积核进行卷积，输出通道数 = 输入通道数，且每个通道的空间尺寸（H/W）可以改变。简单地说就是每个通道配一个自己的卷积核，超参数相同，各算各的。


### 核心入参（复用Conv2d参数规则，默认值同上）
- depthwise层：groups=in_channels，其余参数默认stride=1,padding=0,dilation=1,bias=True
- pointwise层：kernel_size=1，其余默认stride=1,padding=0,groups=1,bias=True
### 优缺点
✅ 参数量仅为标准卷积1/8~1/10，推理速度极快
❌ 特征融合能力弱于标准卷积，精度略有下降
### 适用场景
移动端/嵌入式轻量模型：MobileNet系列、EfficientNet-Lite

In [ ]:
# ====================== 1.2 逐深度卷积 验证 ======================
# 定义输入通道数
in_ch = 64
# 创建Depthwise卷积层：groups=in_ch，每个通道独立卷积，不共享参数
# 注意：bias=False减少参数量，Depthwise层通常不加偏置
depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=3, stride=1, padding=1, groups=in_ch, bias=False)
# 打印参数量：参数量 = in_ch * 1 * 3 * 3，远小于标准卷积的 in_ch * in_ch * 3 * 3
print(f"Depthwise卷积参数量: {sum(p.numel() for p in depthwise.parameters()):,}")
# 对比标准卷积的参数量作为参考
std_conv_ref = nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1)
print(f"同等配置标准卷积参数量: {sum(p.numel() for p in std_conv_ref.parameters()):,}")

## 1.3 分组卷积 Conv2d(groups>1)

分组卷积同时满足「组数等于输入通道、每组仅 1 个卷积核」，等价于逐深度卷积。

### 层定义
标准Conv2d开启groups参数后的变体，将输入、输出通道均等切分为groups组，**组内独立卷积，组间完全隔离**，无信息交互。
### 运算逻辑
输入通道均分为groups份，输出通道同步均分；每组仅和对应分组通道卷积，参数量随groups等比例降低。
特殊值：groups=in_channels 等价于深度可分离卷积的Depthwise层。
### 核心入参
完全复用Conv2d参数列表，groups默认=1，其余参数默认值同Conv2d
### 优缺点
✅ 平衡参数量与精度，可并行计算加速
❌ groups过大时通道交互不足，精度受损
### 适用场景
多分支均衡网络：ResNeXt、ShuffleNet、大通道特征提取

In [ ]:
# ====================== 1.3 分组卷积 验证 ======================
# 创建分组卷积：输入3通道，输出15通道，groups=3
# 每组处理 3/3=1 个输入通道，输出 15/3=5 个通道
# 相当于3组独立的小卷积，组间无信息交互
group_conv = nn.Conv2d(3, 15, kernel_size=3, padding=1, groups=3)
# 使用之前定义的输入x（1,3,32,32）进行前向传播
out_group = group_conv(x)
# 计算并打印参数量
group_params = sum(p.numel() for p in group_conv.parameters())
print(f"GroupConv(groups=3) shape: {x.shape} -> {out_group.shape}")
print(f"GroupConv total params: {group_params:,}")
# 对比标准卷积参数量（同输入输出通道，groups=1）
std_conv_g = nn.Conv2d(3, 15, kernel_size=3, padding=1, groups=1)
print(f"标准卷积(groups=1)参数量: {sum(p.numel() for p in std_conv_g.parameters()):,}")

## 1.4 空洞卷积 Dilated Conv2d

### 层定义
标准Conv2d开启dilation扩张系数，卷积核元素之间插入空白像素，**不增加参数、不降低分辨率，只扩大感受野**。
### 感受野计算公式
$RF = (kernel-1) \times dilation + 1$，dilation=2时3×3核等效5×5感受野。
### 核心入参
完全复用Conv2d参数列表，dilation默认=1，其余参数默认值同Conv2d
### 优缺点
✅ 高密度预测任务扩大全局感受野，无下采样信息丢失
❌ dilation堆叠过大会产生网格效应，远距离像素关联性断裂
### 适用场景
语义分割、全景分割：DeepLabv3+、UperNet

In [ ]:
# ====================== 1.4 空洞卷积 验证 ======================
# 创建空洞卷积：dilation=2，3x3卷积核等效感受野为 5x5
# padding=2 是为了保持输出尺寸不变（输入32x32，输出仍为32x32）
dilated_conv = nn.Conv2d(3, 16, kernel_size=3, padding=2, dilation=2)
# 前向传播
out_dilate = dilated_conv(x)
# 参数量与标准卷积相同（dilation不增加参数）
dilate_params = sum(p.numel() for p in dilated_conv.parameters())
print(f"DilatedConv(dilation=2) shape: {x.shape} -> {out_dilate.shape}")
print(f"DilatedConv total params: {dilate_params:,}")
# 计算等效感受野大小
rf = (3 - 1) * 2 + 1
print(f"3x3卷积核 dilation=2 等效感受野: {rf}x{rf}")

## 1.5 nn.ConvTranspose2d 转置卷积
https://blog.csdn.net/quiet_girl/article/details/84579038
### 层定义
卷积的逆运算，专门用于低分辨率特征图上采样放大空间尺寸，自带可学习卷积参数。有人称其为反卷积（Deconvolution），严格地讲是误称。
### 运算逻辑
输入特征像素之间插入空白0，再执行标准卷积，实现尺寸翻倍放大。
### 核心入参（含默认值）
- in_channels：输入通道【必填】
- out_channels：输出通道【必填】
- kernel_size：卷积核尺寸【必填】
- stride：上采样步长，默认=1
- padding：输入补零，默认=0
- output_padding：输出边缘补零，默认=0
- groups：分组数，默认=1
- bias：可学习偏置，默认=True
- dilation：空洞系数，默认=1
- padding_mode：填充模式，默认='zeros'
### 优缺点
✅ 可学习上采样，适配分割、生成任务特征恢复
❌ 极易出现棋盘格网格伪影，超分辨率任务不推荐
### 适用场景
U-Net分割解码器、GAN图像生成器、图像复原

In [ ]:
# ====================== 1.5 转置卷积 上采样验证 ======================
# 先通过标准卷积得到低分辨率特征图：输入x(1,3,32,32) -> y(1,16,32,32)
y = std_conv(x)
# 创建转置卷积：输入16通道，输出8通道，4x4卷积核，stride=2上采样2倍
# padding=1配合stride=2实现 2x 上采样（32->64）
conv_transpose = nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1)
# 前向传播：输出尺寸 = (32-1)*2 - 2*1 + 4 = 62 + 2 = 64
out_trans = conv_transpose(y)
print(f"ConvTranspose2d shape: {y.shape} -> {out_trans.shape}")
print(f"上采样倍数: {out_trans.shape[2] / y.shape[2]:.0f}x")

## 1.6 nn.PixelShuffle 亚像素上采样

### 层定义
无学习参数的上采样算子（仅通道重组逻辑），把通道维度的像素信息重排到空间维度，替代转置卷积消除棋盘格。
### 通道约束
输入通道数必须等于 $upscale^2 \times out_{channel}$；放大2倍时通道数压缩4倍，空间尺寸翻倍。
### 核心入参（含默认值）
- upscale_factor：放大倍数【必填，无默认】
### 优缺点
✅ 无伪影、无训练参数、推理速度快
❌ 仅固定重组逻辑，无自适应学习能力
### 适用场景
图像超分辨率SRGAN、轻量模型上采样模块

In [ ]:
# ====================== 1.6 PixelShuffle 亚像素上采样验证 ======================
# 创建PixelShuffle层：上采样2倍
pixel_shuffle = nn.PixelShuffle(upscale_factor=2)
# 输入通道数必须是 upscale^2 * out_channels = 4 * 3 = 12
# 输入形状: batch=1, 通道=12, 高=16, 宽=16
x_shuffle = torch.randn(1, 12, 16, 16)
# 前向传播：输出通道压缩为 12/4=3，空间尺寸放大为 16*2=32
out_shuffle = pixel_shuffle(x_shuffle)
print(f"PixelShuffle(2x) shape: {x_shuffle.shape} -> {out_shuffle.shape}")
print(f"通道压缩比: {x_shuffle.shape[1]} / {out_shuffle.shape[1]} = {x_shuffle.shape[1] // out_shuffle.shape[1]}x")

## 1.7 nn.Unfold 图像分块算子

### 层定义
无参数算子，将图像按固定窗口滑动切分为多个Patch张量，等价于手动提取卷积窗口，ViT图像分块专用。
### 输出维度
输出格式 `[batch, patch_total_dim, patch_num]`，patch_total_dim=通道×核高×核宽。
### 核心入参（含默认值）
- kernel_size：窗口尺寸【必填】
- dilation：空洞系数，默认=1
- padding：边缘补零，默认=0
- stride：滑动步长，默认=1
### 优缺点
✅ 快速生成图像序列Patch，适配Transformer输入格式
❌ 仅数据变换，无特征学习能力
### 适用场景
Vision Transformer(ViT)、图像分块预训练

In [ ]:
# ====================== 1.7 Unfold 图像分块验证 ======================
# 创建Unfold层：4x4窗口，步长4（无重叠分块）
unfold = nn.Unfold(kernel_size=(4, 4), stride=4)
# 输入图像：batch=1, 通道=3, 高=32, 宽=32
x_patch = torch.randn(1, 3, 32, 32)
# 前向传播：输出 [batch, patch总维度, patch数量]
# patch总维度 = 通道*核高*核宽 = 3*4*4 = 48
# patch数量 = (32/4)^2 = 64
patches = unfold(x_patch)
# 计算各维度信息
patch_dim = 3 * 4 * 4
patch_num = patches.shape[2]
print(f"Unfold patch shape: {x_patch.shape} -> {patches.shape}")
print(f"Single patch dim: {patch_dim}, total patch count: {patch_num}")
print(f"等价于将 32x32 图像切分为 {int(patch_num**0.5)}x{int(patch_num**0.5)} 个 patch")

## 1.8 边界填充层 ReflectionPad2d / ReplicationPad2d

### 层定义
无参数固定填充算子，卷积前对图像四周补像素，避免卷积后边缘信息丢失。
### 两种填充逻辑
1. ReflectionPad2d：镜像反射填充，边缘像素镜像复制，边界过渡平滑；
2. ReplicationPad2d：复制最外侧边界像素填充，边缘像素保持原始值。
### 核心入参（含默认值）
- padding：单边填充尺寸，int/tuple【必填，无默认】
### 优缺点
Reflection：生成图像边缘更自然；Replication：分类任务稳定无伪影
### 适用场景
Reflection：GAN、图像生成；Replication：图像分类、检测主干网络

In [ ]:
# ====================== 1.8 边界填充层 维度验证 ======================
# 创建镜像反射填充层：上下左右各填充1个像素
reflection_pad = nn.ReflectionPad2d(padding=1)
# 创建复制填充层：复制边缘像素进行填充
replication_pad = nn.ReplicationPad2d(padding=1)
# 对32x32图像进行填充，输出尺寸变为34x34（32+2）
out_reflect = reflection_pad(x)
out_replicate = replication_pad(x)
print(f"ReflectionPad shape: {x.shape} -> {out_reflect.shape}")
print(f"ReplicationPad shape: {x.shape} -> {out_replicate.shape}")
print(f"填充后尺寸变化: 每边增加 {1} 像素")

# 二、线性层 总述

线性层是一维特征映射的基础有参数层，用于将N维特征映射至M维输出，广泛用于分类头、特征融合。
内部可学习参数：权重矩阵W + 偏置b；输入格式 `[batch, feature_dim]`。
包含两类：标准线性nn.Linear、双线性nn.Bilinear。

## 2.1 nn.Linear 标准全连接层

### 层定义
最通用的单路特征线性变换层，完成基础维度映射，CNN、Transformer最后分类头标配。
### 数学公式
$y = xW^T + b$，W为[out_dim, in_dim]权重矩阵，b为输出维度偏置。
### 核心入参（含官方默认值）
- in_features：输入特征维度【必填】
- out_features：输出特征维度【必填】
- bias：是否启用可学习偏置，默认=True
### 优缺点
✅ 任意维度特征映射，表达能力强
❌ 输入维度大时参数量爆炸，易过拟合
### 适用场景
图像分类输出头、特征降维/升维、Transformer首尾映射

In [ ]:
# ====================== 2.1 标准线性层 nn.Linear ======================
# 创建线性层：输入10维，输出5维，默认带偏置
linear = nn.Linear(in_features=10, out_features=5)
# 生成随机输入：batch=4, 特征维度=10
x_linear = torch.randn(4, 10)
# 前向传播：输出 [4, 5]
out_linear = linear(x_linear)
# 获取权重矩阵和偏置的形状
w_shape = linear.weight.shape  # [out_features, in_features] = [5, 10]
b_shape = linear.bias.shape    # [out_features] = [5]
# 计算总参数量：5*10 + 5 = 55
linear_total = sum(p.numel() for p in linear.parameters())
print(f"Linear shape: {x_linear.shape} -> {out_linear.shape}")
print(f"Linear weight shape: {w_shape}, bias shape: {b_shape}")
print(f"Linear total params: {linear_total:,}")
print(f"参数量计算: {w_shape[0]}*{w_shape[1]} + {b_shape[0]} = {linear_total}")

## 2.2 nn.Bilinear 双线性融合层

### 层定义
两路独立输入特征做**二阶交互融合**的有参数层，专门用于捕捉两个不同来源特征向量之间的**交互项（Interaction Term）**。其核心作用是**判断两样东西组合在一起时会产生什么新的"化学反应"，而不是只看它们各自是什么**。

### 数学公式
对于输出维度 $m=1$ 的情况（单标量输出）：
$$ y = x_1^T W x_2 + b $$
其中 $x_1 \in \mathbb{R}^{p}$，$x_2 \in \mathbb{R}^{q}$，$W \in \mathbb{R}^{p \times q}$ 为权重矩阵。

当输出维度 $m > 1$ 时，使用 $m$ 个独立的权重矩阵 $W_1, W_2, \dots, W_m \in \mathbb{R}^{p \times q}$：
$$ y_k = x_1^T W_k x_2 + b_k, \quad k=1,\dots,m $$
整体权重张量形状为 $\mathbb{R}^{m \times p \times q}$。

### 数学本质：交互项的显式建模
以 $p=2, q=2, m=1$ 为例，设 $x_1 = [x_{11}, x_{12}]^T$，$x_2 = [x_{21}, x_{22}]^T$，$W = \begin{bmatrix} w_{11} & w_{12} \\ w_{21} & w_{22} \end{bmatrix}$，展开为：
$$ y = w_{11} \cdot x_{11} x_{21} + w_{12} \cdot x_{11} x_{22} + w_{21} \cdot x_{12} x_{21} + w_{22} \cdot x_{12} x_{22} + b $$

**关键洞察**：展开式中**每一项都是 $x_1$ 中某个元素与 $x_2$ 中某个元素的乘积**（$x_{1i} x_{2j}$），即**纯交互项**，没有任何单独的 $x_{1i}$ 或 $x_{2j}$ 一次项。

对比"拼接 + 线性层"的展开（设拼接后向量为 $[x_{11}, x_{12}, x_{21}, x_{22}]^T$）：
$$ y = w_1 x_{11} + w_2 x_{12} + w_3 x_{21} + w_4 x_{22} + b $$
只有一次项（主效应），**没有任何交互项**——两个输入特征之间完全不交互。

### 与标准线性层的本质区别

| 对比维度 | 拼接 + Linear（单线性融合） | Bilinear（双线性融合） |
|:---|:---|:---|
| 包含的项 | 只有主效应：$\sum_i w_i x_{1i} + \sum_j w_j x_{2j}$ | 只有交互项：$\sum_{i,j} w_{ij} x_{1i} x_{2j}$ |
| 建模能力 | 看"个体"——每个特征独立贡献 | 看"关系"——特征配对后的协同效应 |
| 参数量 | $(p+q) \times m$ | $p \times q \times m$（约为前者的 $\frac{p \times q}{p+q}$ 倍） |

> **关键结论**：双线性层 = **专门为生成并利用"交互项"而设计的层**。如果去掉交互项，它就是一个普通的线性层；加上交互项，它就成了双线性层。

### 与"直接外积+线性压缩"的等价性
外积 $x_1 \otimes x_2 \in \mathbb{R}^{p \times q}$ 后接线性压缩，在数学上**完全等价**于双线性层。但双线性层通过 $W$ 中的 $p \times q$ 个参数，**一步到位**地对所有乘积项进行加权求和，不显式展开外积矩阵，实现更高效的计算。

### 最佳实践：联合单线性与双线性
实际应用中，让模型既看"个体"又看"关系"，通常将两者相加：
$$ y = \underbrace{W^T[x_1; x_2] + b}_{\text{主效应（个体）}} + \underbrace{x_1^T W' x_2}_{\text{交互项（关系）}} $$

### 核心入参（含官方默认值）
- `in1_features`：第一路输入维度 $p$【必填，无默认】
- `in2_features`：第二路输入维度 $q$【必填，无默认】
- `out_features`：融合输出维度 $m$【必填，无默认】
- `bias`：是否启用可学习偏置，默认=`True`

### 优缺点
✅ 天然捕捉跨模态二阶关联，显式建模特征间的**协同/抑制效应**（权重为正→增强，权重为负→抑制）
❌ 权重张量参数量极大（$p \times q \times m$），当 $p, q$ 较大时参数量爆炸

### 适用场景
- **多模态特征融合**：图文匹配、视觉问答（VQA）、音视频融合
- **细粒度图像分类**：局部纹理 × 全局形状的组合特征
- **推荐系统**：用户特征 × 商品特征的交互建模
- **相似度匹配任务**：双塔结构的交互层

> 💡 **降维技巧**：当输入维度很大时，可使用**低秩双线性（Low-rank Bilinear）**近似：$x_1^T W x_2 \approx (U^T x_1)^T (V^T x_2)$，将参数量从 $p \times q \times m$ 降为 $(p \times r) + (q \times r) + (r \times m)$。

In [ ]:
# ====================== 2.2 双线性层 nn.Bilinear ======================
# 创建双线性层：第一路输入10维，第二路输入8维，输出5维
# 权重张量形状：[out_features, in1_features, in2_features] = [5, 10, 8]
bilinear = nn.Bilinear(in1_features=10, in2_features=8, out_features=5)
# 生成两路随机输入
x1 = torch.randn(4, 10)
x2 = torch.randn(4, 8)
# 前向传播：输出 [4, 5]
out_bi = bilinear(x1, x2)
print(f"Bilinear output shape: {out_bi.shape}")
print(f"Bilinear weight shape: {bilinear.weight.shape}")
print(f"Bilinear bias shape: {bilinear.bias.shape}")

# ====================== 额外验证：参数量对比 ======================
# 对比"拼接+线性"与"双线性"的参数量
p, q, m = 10, 8, 5
# 拼接+线性：将两路拼成 p+q 维，再映射到 m 维
linear_combined_params = (p + q) * m + m
# 双线性：p*q*m 个权重 + m 个偏置
bilinear_params = p * q * m + m
print(f"\n参数量对比 (p={p}, q={q}, m={m}):")
print(f"  拼接+线性: {linear_combined_params:,}")
print(f"  双线性:   {bilinear_params:,}")
print(f"  比例:     {bilinear_params / linear_combined_params:.2f}x")
print(f"  双线性参数量是拼接+线性的 {bilinear_params / linear_combined_params:.1f} 倍")

# 三、归一化层 总述

归一化层是带可学习缩放 γ（Gamma）、偏移 β（Beta）的有参数层。这意味着 γ 和 β **不是人工设定的固定常数**（如永远等于 1 和 0），而是**像卷积核权重一样，在反向传播过程中通过梯度下降算法自动迭代更新**。

- **γ（缩放）**：控制数据的"幅度"（方差/范围）。
- **β（偏移）**：控制数据的"基线"（均值/中心点）。

> 📌 **关键理解：γ 和 β 为什么不是"白做"？**
> 
> 初学者常问："归一化把数据拉到 0 均值，γ/β 又把它变回去，这不是白做了吗？"
> 
> 答案在于**训练阶段与推理阶段的分工**，以及**稳定性与表达力的权衡**：
> 
> - **训练前期（强制稳定）**：权重随机，数据分布剧烈抖动。强制归一化将数据限制在 0 均值附近，有效防止梯度爆炸/消失，**保证训练顺利启动**。
> - **训练后期（自主调节）**：网络在平稳环境下逐渐收敛。若网络发现"保持原始数据幅度"能获得更低的损失，它可以通过调整 γ 和 β 来**近似复原原始分布**——但此时模型已经平稳收敛，它是在"安全环境下"逐步恢复的，而不是一开始就没做归一化。
> 
> **核心逻辑**：
> 1. **撤销的权限 ≠ 撤销的必要性**——给了网络权限，不代表它一定会撤销，而是赋予它"选择最优分布范围"的自由。
> 2. **即使数学上完全复原（γ = 标准差，β = 均值），也≠没做归一化**：因为反向传播时，梯度流经的输入已经过"去相关"处理（除以标准差），消除了不同维度间的尺度差异，这种良好的优化状态会持续影响整个训练过程。
> 
> **直观类比（烹饪）**：
> - **归一化（焯水）**：去除杂质，让所有食材处于统一的基础状态，便于后续处理。
> - **γ/β（调味）**：盐和酱油放多少，不是厨师拍脑袋定的，而是根据食客反馈（损失函数）在训练中不断调整出的"最佳口味"。
> 
> **核心结论**：归一化层 = 强制标准化（保证训练稳定性） + 可学习仿射变换（保留模型容量）。引入 γ 和 β 是给网络增加了一个**"刻度调节器"**，让网络既能享受标准化带来的优化平滑性，又能拥有自主调节特征分布的权力，在**训练稳定性**与**特征表达能力**之间达到绝佳平衡。

归一化层核心作用：标准化特征分布、加速训练、缓解梯度消失。训练时基于指定维度计算均值方差标准化，再通过 γ、β 恢复特征表达能力。γ 和 β 是**逐通道（Channel-wise）**的——例如卷积层输出 64 个通道，则该归一化层对应有 64 个 γ 和 64 个 β。

分为批次归一化、样本归一化、通道分组归一化、权重约束归一化四大类。

## 3.1 BatchNorm2d 批次归一化

### 层定义
CNN最经典归一化，基于**整个batch样本**对单通道特征做标准化。
### 数学逻辑（训练阶段）
$$\mu = \frac{1}{N}\sum_{batch} x, \quad \sigma^2 = \frac{1}{N}\sum_{batch}(x-\mu)^2$$
$$\hat{x} = \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}, \quad y = \gamma \hat{x} + \beta$$
γ、β为可学习参数，推理时使用训练滑动平均均值方差。
### 核心入参（含默认值）
- num_features：输入通道数C【必填】
- eps：数值稳定极小值，默认=1e-5
- momentum：滑动平均更新系数，默认=0.1
- affine：是否启用可学习γ/β，默认=True
- track_running_stats：是否跟踪全局均值方差，默认=True
### 优缺点
✅ 大幅加速收敛，稳定训练分布，正则化效果
❌ batch_size极小时统计量偏差大，性能暴跌
### 适用场景
batch≥16的图像分类、检测CNN主干

In [ ]:
# ====================== 3.1 BatchNorm2d 验证 ======================
# 创建BatchNorm2d：输入通道数为8
# affine=True（默认）表示启用可学习的γ和β
bn = nn.BatchNorm2d(num_features=8)
# 生成随机输入：batch=4, 通道=8, 高=16, 宽=16
x_bn = torch.randn(4, 8, 16, 16)
# 前向传播：输出与输入shape相同
out_bn = bn(x_bn)
print(f"BatchNorm2d input shape: {x_bn.shape} -> output shape: {out_bn.shape}")
# γ（weight）和β（bias）是逐通道的，shape均为 [num_features]
print(f"BatchNorm2d 可学习参数 γ (weight) shape: {bn.weight.shape}")
print(f"BatchNorm2d 可学习参数 β (bias) shape: {bn.bias.shape}")
# 验证归一化效果：输出应接近均值为0、标准差为1
print(f"BatchNorm2d 输出均值: {out_bn.mean().item():.4f}, 标准差: {out_bn.std().item():.4f}")
print(f"注：由于batch size=4较小，统计量有轻微偏差，但仍接近0和1")

## 3.2 LayerNorm 层归一化

### 层定义
不依赖批次，**单样本内部全部特征维度**做标准化，Transformer标配归一化。
### 核心特点
均值方差在单个样本内计算，batch_size无任何约束，时序、文本任务稳定。
### 核心入参（含默认值）
- normalized_shape：归一化维度尺寸【必填】
- eps：数值稳定极小值，默认=1e-5
- elementwise_affine：启用可学习γ/β，默认=True
### 优缺点
✅ 不限batch大小，NLP/Transformer稳定
❌ CNN大batch场景收敛速度弱于BN
### 适用场景
BERT、GPT、ViT、所有Transformer架构、小batch时序模型

In [ ]:
# ====================== 3.2 LayerNorm 验证 ======================
# 创建LayerNorm：指定需要归一化的维度 [C, H, W] = [8, 16, 16]
# 每个样本的这8*16*16=2048个元素作为一个整体进行标准化
ln = nn.LayerNorm(normalized_shape=[8, 16, 16])
# 使用相同的输入x_bn（4, 8, 16, 16）
x_ln = torch.randn(4, 8, 16, 16)
out_ln = ln(x_ln)
print(f"LayerNorm input shape: {x_ln.shape} -> output shape: {out_ln.shape}")
# γ和β的shape等于normalized_shape，即 [8, 16, 16]
print(f"LayerNorm 可学习参数 γ (weight) shape: {ln.weight.shape}")
print(f"LayerNorm 可学习参数 β (bias) shape: {ln.bias.shape}")
# LayerNorm不依赖batch，每个样本独立归一化
# 验证：所有样本的均值应为0，标准差为1
print(f"LayerNorm 输出全局均值: {out_ln.mean().item():.4f}, 标准差: {out_ln.std().item():.4f}")
# 检查每个样本的均值是否接近0
sample_means = out_ln.mean(dim=[1, 2, 3])
print(f"各样本均值: {sample_means.tolist()}")

## 3.3 InstanceNorm2d 实例归一化

### 层定义
InstanceNorm2d 将每个样本的每个通道视为一个独立的"实例"，**统计量（均值和方差）在单个样本的单个通道内独立计算**，完全抛弃批次维度的统计信息。



### 核心作用
消除图像间的对比度、亮度差异（风格信息），保留内容结构。

### 核心入参（含默认值）
- num_features：输入通道数 C【必填】
- eps：数值稳定极小值，默认=1e-5
- affine：是否启用可学习 γ/β，默认=`False` ⚠️ **注意：与 BN 不同，默认关闭，如需参数需手动设为 True**
- track_running_stats：是否跟踪全局统计，默认=False

### 优缺点
✅ 消除图像固有风格信息，风格迁移效果极佳
✅ 完全不受 batch size 影响，batch=1 时依然有效
❌ 分类任务丢失批次全局统计，精度下降

### 适用场景
图像风格迁移、CycleGAN、图像生成

In [ ]:
# ====================== 3.3 InstanceNorm2d 验证 ======================
# 创建InstanceNorm2d：输入通道数为8
# affine=True 启用可学习的γ和β（注意：官方默认affine=False）
in_norm = nn.InstanceNorm2d(num_features=8, affine=True)
# 使用相同的输入
x_in = torch.randn(4, 8, 16, 16)
out_in = in_norm(x_in)
print(f"InstanceNorm2d input shape: {x_in.shape} -> output shape: {out_in.shape}")
# γ和β是逐通道的，shape为 [num_features]
print(f"InstanceNorm2d 可学习参数 γ (weight) shape: {in_norm.weight.shape}")
print(f"InstanceNorm2d 可学习参数 β (bias) shape: {in_norm.bias.shape}")
# InstanceNorm在每个样本的每个通道上独立做归一化
# 检查第1个样本第1个通道的均值和标准差
ch0_mean = out_in[0, 0].mean().item()
ch0_std = out_in[0, 0].std().item()
print(f"样本0通道0 均值: {ch0_mean:.4f}, 标准差: {ch0_std:.4f}")
print("每个样本每个通道独立归一化，因此全局均值和标准差并不严格为0和1")

## 3.4 GroupNorm 分组归一化

### 层定义
GroupNorm 是 BN 和 IN 的折中方案，将通道分为多组，**统计量（均值和方差）在每组的每个通道内部独立计算**，完全不依赖 batch 维度。



### 🤔 那 GroupNorm 和 InstanceNorm 到底有什么区别？

这是一个非常关键的问题！

**如果统计量计算方式完全一样（都是 `[H, W]`），那 GroupNorm 和 InstanceNorm 的区别在哪里？**

**答案：训练时的梯度传播和正则化效果不同！**

| 对比维度 | InstanceNorm | GroupNorm |
|:---|:---|:---|
| 统计量计算 | 每个通道独立 `[H, W]` | 每个通道独立 `[H, W]` |
| **分组的作用** | 无分组概念 | 同一组内的通道在**反向传播时共享信息** |
| **梯度传播** | 每个通道完全独立 | 组内通道的梯度相互影响 |
| **正则化效果** | 每个通道独立做归一化 | 组内通道相互约束，具有协同正则化效果 |

> 💡 **本质理解**：
> 
> GroupNorm 的分组**不是改变统计量的计算方式**（统计量始终是逐通道的 `[H, W]`），而是**改变网络的学习方式**。
> 
> - 同一组内的通道被"绑定"在一起：它们共享同一种归一化"节奏"
> - 这相当于一种**结构化约束**：告诉网络"这组通道应该表现出相似的数据分布"
> - 对于小 batch 训练，这种约束比完全独立的 InstanceNorm 更有利于收敛

### 核心入参（含默认值）
- num_groups：分组数量【必填】
- num_channels：总通道数 C【必填】
- eps：数值稳定极小值，默认=1e-5
- affine：启用可学习 γ/β，默认=`True`

### 优缺点
✅ 小 batch（1/2/4）下性能远超 BN，无需多卡同步
✅ 不受 batch size 限制
❌ 大 batch 场景收敛速度略慢于 BN

### 适用场景
分割、检测小 batch 训练、医疗影像模型

In [ ]:
# ====================== 3.4 GroupNorm 验证 ======================
# 创建GroupNorm：将8个通道分为4组，每组2个通道
# 每组内部独立计算均值和方差
gn = nn.GroupNorm(num_groups=4, num_channels=8)
# 使用相同的输入
x_gn = torch.randn(4, 8, 16, 16)
out_gn = gn(x_gn)
print(f"GroupNorm input shape: {x_gn.shape} -> output shape: {out_gn.shape}")
# γ和β是逐通道的，shape为 [num_channels]
print(f"GroupNorm 可学习参数 γ (weight) shape: {gn.weight.shape}")
print(f"GroupNorm 可学习参数 β (bias) shape: {gn.bias.shape}")
print(f"GroupNorm 分组数: {gn.num_groups}, 每组通道数: {gn.num_channels // gn.num_groups}")
# 验证：输出应接近0均值、1标准差
print(f"GroupNorm 输出均值: {out_gn.mean().item():.4f}, 标准差: {out_gn.std().item():.4f}")

## 3.5 SyncBatchNorm 多卡同步批次归一化

### 层定义
分布式DDP训练专用BN变体，跨多张GPU同步全部batch样本的均值方差。
### 核心特点
单卡batch很小时，多卡合并全局统计量，等效扩大有效batch尺寸。
### 核心入参
底层参数与BatchNorm2d完全一致，默认值相同；通过SyncBatchNorm.convert_sync_batchnorm()转换实例
### 适用场景
多GPU分布式训练、大图像分割任务

In [ ]:
# ====================== 3.5 SyncBatchNorm 验证 ======================
# 先创建一个普通的BatchNorm2d实例
bn_for_sync = nn.BatchNorm2d(num_features=8)
# 通过convert_sync_batchnorm转换为SyncBatchNorm
# 在分布式训练中，SyncBatchNorm会跨GPU同步统计量
sync_bn = nn.SyncBatchNorm.convert_sync_batchnorm(bn_for_sync)
# 使用相同的输入
x_sync = torch.randn(4, 8, 16, 16)
# 注意：单卡运行时，SyncBatchNorm行为与BatchNorm2d相同
out_sync = sync_bn(x_sync)
print(f"SyncBatchNorm input shape: {x_sync.shape} -> output shape: {out_sync.shape}")
print(f"SyncBatchNorm 可学习参数 γ (weight) shape: {sync_bn.weight.shape}")
print(f"SyncBatchNorm 可学习参数 β (bias) shape: {sync_bn.bias.shape}")
print(f"SyncBatchNorm 输出均值: {out_sync.mean().item():.4f}, 标准差: {out_sync.std().item():.4f}")
print("注：多GPU训练时，SyncBatchNorm会同步所有卡的统计量，等效于大batch BN")

## 3.6 SpectralNorm 谱归一化

https://blog.csdn.net/shizheng_Li/article/details/146983944

### 层定义
谱归一化是一种权重约束型归一化，作为包装器包裹 Conv2d / Linear 等层。其核心操作是**将权重矩阵除以自身的最大奇异值**，强制**谱范数（最大奇异值）≤ 1**。

> 📌 **什么是"谱"？**
>
> 在线性代数中，矩阵的"谱"指**所有奇异值（Singular Values）的集合**：
> $$\sigma_1 \ge \sigma_2 \ge \sigma_3 \ge \dots \ge \sigma_r \ge 0$$
> 
- 谱范数 = 最大奇异值$\sigma_1$，代表该矩阵的最大放大倍数- 
- 谱归一化 = 对权重矩阵进行 $\sigma_1$ 约束：$ W_{SN} = W / \sigma_1(W) $

由于直接计算完整 SVD 分解计算量过大，PyTorch 使用**幂迭代法**高效近似求解 $\sigma_1$
>
> **直观理解**：如果输入信号变化 0.1，权重矩阵最大放大倍数为 2，则输出最多变化 0.2；谱归一化把最大放大倍数限制为 1，输出最多变化 0.1——网络对输入噪声不再过度敏感。

### 核心作用

| 作用 | 说明 |
|:---|:---|
| **Lipschitz 约束** | 强制每层网络的 Lipschitz 常数为 1，即网络对输入的微小变化不会产生剧烈反应 |
| **稳定 GAN 训练** | 防止判别器过于"强势"，避免生成器梯度消失，使对抗训练更平衡 |
| **缓解模式崩溃** | 约束使损失函数更平滑，优化过程更稳定，减少生成样本单一化 |





### 使用方式

```python
nn.utils.spectral_norm(module, n_power_iterations=1, eps=1e-12)
```

- n_power_iterations：幂迭代次数--每次前向传播时，幂迭代（Power Iteration）执行的步数。
- eps：eps 是一个很小的数，用于防止分母为零。

In [ ]:
# ====================== 3.6 谱归一化验证（多次迭代版） ======================
import torch
import torch.nn as nn

# 创建谱归一化层
sn_linear = nn.utils.spectral_norm(nn.Linear(10, 10))

# 固定输入用于一致性
x_dummy = torch.randn(4, 10)

print("=== 迭代过程中谱范数变化 ===")
print("iter 0 (init):", torch.linalg.matrix_norm(sn_linear.weight, ord=2).item())

for i in range(1, 100):
    _ = sn_linear(x_dummy)  # 前向传播 → 触发 u, v 更新
    norm = torch.linalg.matrix_norm(sn_linear.weight, ord=2).item()
    print(f"iter {i}: {norm:.6f}")

## 3.7 WeightNorm 权重归一化

### 层定义
权重归一化（Weight Normalization, WN）是一种重参数化技术，将原始权重矩阵 $W$ 拆解为**方向向量 $V$** 和**缩放标量 $g$** 的乘积：

$$W = g \cdot \frac{V}{\|V\|_2}$$

其中 $\frac{V}{\|V\|_2}$ 是单位方向向量（决定权重的方向），$g$ 是标量（决定权重的长度/幅度），两者均为可训练参数。

---

### 权重归一化的本质作用

WN 的核心目标是**改善优化景观**——通过将权重解耦为"方向"和"大小"两个独立分量，减轻权重尺度对梯度的敏感性，从而加速收敛。

#### 解耦带来的优化优势

| 对比维度 | 原始权重 $W$ | 权重归一化 $W = g \cdot \hat{V}$ |
|:---|:---|:---|
| 梯度结构 | $\nabla_W L$ 直接依赖 $W$ 的尺度 | 梯度被投影到与 $V$ 正交的方向上 |
| 尺度敏感性 | 权重越大，梯度范数通常越大 | 梯度与权重尺度解耦，更稳定 |
| 优化条件数 | 受权重尺度不均影响 | 有效降低曲率，加速收敛 |

#### WN 的梯度投影效应（数学直观）

对于方向向量 $V$ 的梯度：

$$\nabla_V L = \frac{g}{\|V\|} \left( I - \hat{V}\hat{V}^T \right) \nabla_W L$$

其中 $\left( I - \hat{V}\hat{V}^T \right)$ 是**投影矩阵**，它将 $\nabla_W L$ 投影到与 $V$ 正交的方向上。

> 💡 **这意味着**：WN 不仅仅是"缩放梯度的大小"，而是**改变了梯度的方向结构**——这是它与调整学习率的根本区别之一。

---

### ⚠️ 权重归一化 ≠ 逐参数学习率

尽管两者在"控制参数更新幅度"这个表象上有相似之处，但它们的**数学本质、作用机制和设计目标**完全不同：

| 对比维度 | 权重归一化 (WN) | 调整学习率 (LR) |
|:---|:---|:---|
| **作用对象** | 单个参数向量（逐神经元/逐通道） | 全体参数（全局标量） |
| **粒度** | 细粒度，每个神经元独立调控 | 粗粒度，所有参数共享同一缩放系数 |
| **调控方式** | **参数重参数化**（改变优化空间的几何结构） | **优化器超参数**（缩放梯度更新量） |
| **是否可学习** | ✅ `g` 和 `V` 通过梯度下降自动更新 | ❌ 超参数，人为设定或调度器控制 |
| **是否改变梯度方向** | ✅ 是（通过正交投影 $(I - \hat{V}\hat{V}^T)$） | ❌ 否（只缩放，不转向） |
| **是否改变优化几何** | ✅ 是（重参数化改变了损失景观） | ❌ 否（在原空间调整步长） |
| **设计目标** | 改善优化景观——降低曲率，加速收敛 | 平衡收敛速度与稳定性 |

> 📌 **最精确的理解**：
> 
- 权重归一化通过**重参数化**解耦权重的方向与大小，改善优化几何条件；而调整学习率是在原参数空间**简单缩放**梯度步长。
  - 重参数化（Reparameterization） = 重新选择一套参数来表达同一个东西。
- 如果你把 WN 理解为"为每个神经元单独设置了一个可学习的动态学习率"，这在直觉上接近，但在数学上不严谨——因为 WN 改变的是参数化方式本身，而不仅仅是更新步长。

---

### 使用方式

```python
nn.utils.weight_norm(module, name='weight', dim=0)
```

- `name`：权重参数名，默认=`'weight'`
- `dim`：归一化维度，默认=`0`

---

### 优缺点

- ✅ 改善权重矩阵条件数，加速深层网络收敛
- ✅ 不依赖批次统计，适用于小 batch 场景
- ❌ 不改变权重范数本身，对过拟合无直接约束效果
- ❌ 在 BatchNorm 同时使用时效果可能叠加或相互干扰

---

### 适用场景

- 深层 CNN（替代或补充 BN）
- 小 batch 时序模型
- RNN / LSTM 训练（减少梯度爆炸）
- 生成模型（配合 GAN 使用）

In [ ]:
# ====================== 3.7 权重归一化验证 ======================
# 权重归一化：将权重分解为方向（weight_v）和大小（weight_g）两部分
# weight = weight_g * weight_v / ||weight_v||
# 这使得梯度可以分别优化方向和大小，改善条件数
wn_linear = nn.utils.weight_norm(nn.Linear(10, 10))
# 生成随机输入
x_wn = torch.randn(4, 10)
out_wn = wn_linear(x_wn)
print(f"WeightNorm wrapped Linear input shape: {x_wn.shape} -> output shape: {out_wn.shape}")
print(f"WeightNorm wrapped layer class: {wn_linear.__class__.__name__}")
# weight_g: 缩放标量（大小），shape为 [out_features]
print(f"WeightNorm 原始 weight_g (缩放标量g) shape: {wn_linear.weight_g.shape}")
# weight_v: 方向向量，shape与原始weight相同
print(f"WeightNorm 原始 weight_v (方向向量v) shape: {wn_linear.weight_v.shape}")
# 打印前3个缩放标量的值（而不是全部10个）
print(f"WeightNorm weight_g 前3个值: {wn_linear.weight_g[:3].tolist()}")
print("权重归一化解耦了权重的方向和大小，使优化更稳定、收敛更快")

# 四、嵌入层 总述

嵌入层是离散编码专用有参数层，维护一张可学习查找表，将离散整数索引映射为稠密低维连续向量。
内部可学习参数：词向量查找表weight矩阵；主要用于NLP离散文本、分类离散特征编码。

## 4.1 nn.Embedding 基础词嵌入层

### 层定义
基础查找嵌入层，输入整数索引序列，查表输出对应稠密向量。
### 核心入参（含默认值）
- num_embeddings：词典总大小【必填】
- embedding_dim：输出向量维度【必填】
- padding_idx：padding索引，默认=None
- max_norm：词向量最大范数约束，默认=None
- norm_type：范数类型，默认=2.0
- scale_grad_by_freq：梯度按词频缩放，默认=False
- sparse：稀疏梯度模式，默认=False
### 运算逻辑
输入整数直接作为weight矩阵行下标，取出对应向量，无矩阵乘法。
### 适用场景
文本词编码、分类离散类别特征映射、推荐系统ID嵌入

In [ ]:
# ====================== 4.1 Embedding 验证 ======================
# 创建Embedding层：词典大小1000，每个词映射为300维向量
embedding = nn.Embedding(num_embeddings=1000, embedding_dim=300)
# 输入：整数索引序列，shape [batch, seq_len] = [2, 3]
# 每个整数对应词典中的一个词
x_indices = torch.tensor([[1, 2, 3], [4, 5, 6]])
# 前向传播：查表得到词向量，输出 [2, 3, 300]
emb_out = embedding(x_indices)
print(f"Embedding input indices shape: {x_indices.shape} -> output {emb_out.shape}")
# 可学习参数：词向量查找表，shape [num_embeddings, embedding_dim]
print(f"Embedding weight table shape: {embedding.weight.shape}")
print(f"词表大小: {embedding.num_embeddings}, 每个词向量维度: {embedding.embedding_dim}")
print("注意：Embedding层没有矩阵乘法，只是查表操作")

## 4.2 nn.EmbeddingBag 聚合嵌入层

### 层定义
内置聚合逻辑的增强嵌入层，单次完成查表+池化聚合，无需手动拼接求和。
### 核心入参（含默认值）
- num_embeddings：词典大小【必填】
- embedding_dim：输出向量维度【必填】
- mode：聚合模式，可选sum/mean/max，默认='mean'
- padding_idx：padding索引，默认=None
- max_norm：范数约束，默认=None
- norm_type：范数类型，默认=2.0
- scale_grad_by_freq：梯度缩放，默认=False
- sparse：稀疏梯度，默认=False
- include_last_offset：是否包含末尾偏移，默认=False
### 优势
减少中间张量显存占用，简化变长文本池化代码。
### 适用场景
句子分类、文档向量编码、多ID特征聚合

In [ ]:
# ====================== 4.2 EmbeddingBag 验证 ======================
# 创建EmbeddingBag：mode='mean'表示对每个bag内的词向量取平均
emb_bag = nn.EmbeddingBag(num_embeddings=1000, embedding_dim=300, mode='mean')
# 输入：所有样本的所有词索引拼接成一个一维张量
indices = torch.tensor([1, 2, 3, 4, 5])
# offsets：每个bag的起始位置，第一个bag从0开始，第二个从3开始
# 即 bag1: [1,2,3], bag2: [4,5]
offsets = torch.tensor([0, 3])
# 前向传播：自动完成查表+聚合，输出 [2, 300]
bag_out = emb_bag(indices, offsets)
print(f"EmbeddingBag input indices: {indices.tolist()}, offsets: {offsets.tolist()}")
print(f"EmbeddingBag output shape (batch=2): {bag_out.shape}")
print(f"Bag1 包含词索引 [1,2,3]，Bag2 包含 [4,5]")
print(f"每个bag输出一个300维的聚合向量 (mode='mean' 取平均)")

# 五、循环序列层 总述

循环层用于时序/文本序列建模，带可学习门控权重，逐时间步迭代提取时序依赖。
分为高层封装RNN/LSTM/GRU、单步Cell、变长打包工具三类，全部包含可训练权重与偏置。

## 5.1 nn.RNN 基础循环网络

### 层定义
最简单的时序循环层，仅单一线性门控传递隐态，梯度极易长距离消失。
### 运算逻辑
当前输入特征 + 上一时刻隐态拼接，线性变换+激活得到新隐态，无遗忘门控。
### 核心入参（含默认值）
- input_size：输入特征维度【必填】
- hidden_size：隐态维度【必填】
- num_layers：堆叠层数，默认=1
- nonlinearity：激活函数tanh/relu，默认='tanh'
- bias：是否启用偏置，默认=True
- batch_first：输入格式(batch,seq,feat)，默认=False
- dropout：层间dropout概率，默认=0.0
- bidirectional：双向循环，默认=False
### 优缺点
✅ 参数量最小、推理最快
❌ 长序列梯度消失严重，无法捕捉远距离依赖
### 适用场景
极短时序预测、简单分类任务

In [ ]:
# ====================== 5.1 RNN 验证 ======================
# 创建双向RNN：输入10维，隐态20维，2层，batch_first=True
rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=2, batch_first=True, bidirectional=True)
# 生成随机序列：batch=4, seq_len=16, input_size=10
x_rnn = torch.randn(4, 16, 10)
# 前向传播：输出所有时间步的隐态 [4, 16, 40] (双向: 20*2)
out_rnn, h_rnn = rnn(x_rnn)
print(f"BiRNN input shape: {x_rnn.shape} -> output shape: {out_rnn.shape}")
# h_rnn: 最后一层所有方向的最终隐态 [num_layers*2, batch, hidden_size]
print(f"BiRNN final hidden state shape: {h_rnn.shape}")
print(f"输出维度 = hidden_size * (2 if bidirectional else 1) = {out_rnn.shape[-1]}")

## 5.2 nn.LSTM 长短期记忆网络

### 层定义
引入三门控（输入门、遗忘门、输出门）+细胞态cell state，解决长序列梯度消失。
### 核心机制
细胞态线性信息流传递，遗忘门控制历史信息保留，输入门控制当前信息写入。
### 核心入参（含默认值）
- input_size：输入特征维度【必填】
- hidden_size：隐态维度【必填】
- num_layers：堆叠层数，默认=1
- bias：启用偏置，默认=True
- batch_first：batch优先格式，默认=False
- dropout：层间dropout，默认=0.0
- bidirectional：双向，默认=False
- proj_size：输出投影维度，默认=0（无投影）
### 优缺点
✅ 长文本、长时序依赖捕捉能力极强
❌ 四层门控，参数量大，推理速度慢
### 适用场景
机器翻译、语音识别、长文本分类、时间序列预测

In [ ]:
# ====================== 5.2 LSTM 验证 ======================
# 创建双向LSTM：输入10维，隐态20维，2层，batch_first=True
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True, bidirectional=True)
# 生成随机序列
x_lstm = torch.randn(4, 16, 10)
# 前向传播：LSTM返回输出、隐态、细胞态
out_lstm, (h_lstm, c_lstm) = lstm(x_lstm)
print(f"BiLSTM input shape: {x_lstm.shape} -> output shape: {out_lstm.shape}")
# h_lstm: 最终隐态，c_lstm: 最终细胞态
print(f"BiLSTM final hidden state shape: {h_lstm.shape}")
print(f"BiLSTM final cell state shape: {c_lstm.shape}")
print("LSTM通过细胞态线性信息流和门控机制，解决了长序列梯度消失问题")

## 5.3 nn.GRU 门控循环单元

### 层定义
LSTM轻量化变体，合并遗忘门与输入门为更新门，取消独立细胞态。
### 核心机制
仅更新门、重置门两个门控，隐态直接承载全部时序信息。
### 核心入参（含默认值）
参数列表、默认值与LSTM完全一致，无proj_size参数
### 优缺点
✅ 参数量、速度介于RNN与LSTM之间，精度接近LSTM
### 适用场景
资源受限设备时序模型、文本分类、对话系统

In [ ]:
# ====================== 5.3 GRU 验证 ======================
# 创建GRU：输入10维，隐态20维，2层，batch_first=True
# GRU比LSTM少一个细胞态，参数量更少
gru = nn.GRU(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
x_gru = torch.randn(4, 16, 10)
out_gru, h_gru = gru(x_gru)
print(f"GRU input shape: {x_gru.shape} -> output shape: {out_gru.shape}")
print(f"GRU final hidden state shape: {h_gru.shape}")
print("GRU仅有更新门和重置门，是LSTM的轻量级替代方案")

## 5.4 LSTMCell / RNNCell 单步循环单元

### 层定义
无自动时间步循环的底层单元，仅处理单个时间步输入，需要手动for循环遍历序列。
### LSTMCell核心入参（默认值）
- input_size【必填】、hidden_size【必填】、bias=True
### RNNCell核心入参（默认值）
- input_size【必填】、hidden_size【必填】、bias=True、nonlinearity='tanh'
### 核心用途
自回归生成、强制解码、自定义时序迭代逻辑（逐词生成文本）。

In [ ]:
# ====================== 5.4 LSTMCell 验证 ======================
# 创建LSTMCell：输入10维，隐态20维
# Cell只处理单步，需要手动循环
lstm_cell = nn.LSTMCell(input_size=10, hidden_size=20)
# 生成序列数据：batch=4, seq_len=16, input_size=10
x_cell = torch.randn(4, 16, 10)
# 初始化隐态h和细胞态c为全0
h = torch.zeros(4, 20)
c = torch.zeros(4, 20)
# 手动遍历每个时间步
for t in range(x_cell.size(1)):
    # 每一步输入当前时刻的特征，更新隐态和细胞态
    h, c = lstm_cell(x_cell[:, t, :], (h, c))
print(f"LSTMCell input seq_len=16, batch=4, input_size=10")
print(f"LSTMCell final hidden state shape: {h.shape}")
print(f"LSTMCell final cell state shape: {c.shape}")
print("LSTMCell需要手动循环，适用于自回归生成等自定义迭代场景")

## 5.5 PackedSequence 变长序列打包工具

### 层定义
无参数工具，对不等长序列打包，跳过padding零值区域计算，节省显存与算力。
### pack_padded_sequence核心参数
- input：填充后序列【必填】
- lengths：每条序列真实长度【必填】
- batch_first：默认=False
- enforce_sorted：是否强制序列降序排列，默认=True
### pad_packed_sequence核心参数
- sequence：打包序列【必填】
- batch_first：默认=False
- padding_value：填充值，默认=0.0
- total_length：输出固定长度，默认=None
### 配套函数
pack_padded_sequence：打包变长序列；pad_packed_sequence：解压恢复对齐张量。
### 适用场景
文本、语音不等长序列训练，消除padding冗余计算。

In [ ]:
# ====================== 5.5 PackedSequence 验证 ======================
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 生成填充后的序列：batch=4, seq_len=16, input_size=10
x_pack = torch.randn(4, 16, 10)
# 定义每个样本的真实长度（不同样本长度不同）
seq_lengths = torch.tensor([10, 8, 12, 6])

# 打包：将变长序列打包，跳过padding部分的计算
# enforce_sorted=False允许输入不按长度降序排列
packed = pack_padded_sequence(x_pack, seq_lengths, batch_first=True, enforce_sorted=False)
# 将打包序列输入LSTM
lstm_pack = nn.LSTM(10, 20, batch_first=True)
out_packed, _ = lstm_pack(packed)
# 解包：恢复为填充后的对齐张量
out_unpacked, lengths = pad_packed_sequence(out_packed, batch_first=True)

print(f"PackedSequence 原始输入 shape: {x_pack.shape}")
print(f"PackedSequence 各样本真实长度: {seq_lengths.tolist()}")
print(f"PackedSequence 解压后输出 shape: {out_unpacked.shape}")
print(f"PackedSequence 解压后各样本实际长度: {lengths.tolist()}")
print("PackedSequence通过跳过padding计算，节省了约 30%-50% 的计算量")

# 六、Transformer 核心组件 总述

Transformer基于自注意力机制建模全局序列依赖，全部组件均包含可学习参数，是NLP、ViT基础架构。
核心模块：多头注意力MHA、编码器层、解码器层、完整Transformer容器。

## 6.1 nn.MultiheadAttention 多头注意力

### 层定义
自注意力核心有参数层，将特征切分为多头并行计算注意力，捕捉多尺度依赖。
### 运算逻辑
输入分别通过Q/K/V线性投影矩阵，分头计算缩放点积注意力，拼接多头输出后线性融合。
### 核心入参（含默认值）
- embed_dim：特征总维度【必填】
- num_heads：注意力头数量【必填】
- dropout：注意力权重dropout，默认=0.0
- bias：输出线性层偏置，默认=True
- add_bias_kv：给K/V添加可学习偏置，默认=False
- add_zero_attn：补充零注意力token，默认=False
- kdim：K输入维度，默认=None（等于embed_dim）
- vdim：V输入维度，默认=None（等于embed_dim）
- batch_first：batch优先格式，默认=False
### 适用场景
所有Transformer架构、ViT视觉自注意力

In [ ]:
# ====================== 6.1 MultiheadAttention 验证 ======================
# 创建多头注意力：特征维度512，8个注意力头，batch_first=True
mha = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)
# 生成输入：batch=4, seq_len=32, embed_dim=512
x_mha = torch.randn(4, 32, 512)
# 自注意力：Q=K=V=x_mha
attn_out, attn_weights = mha(x_mha, x_mha, x_mha)
print(f"MultiheadAttention input shape: {x_mha.shape} -> output shape: {attn_out.shape}")
# 注意力权重矩阵：batch=4, num_heads=8, query_len=32, key_len=32
print(f"Attention weight matrix shape: {attn_weights.shape}")
print(f"每个头负责 {512//8} 维度的特征，多头并行捕捉不同子空间特征")

## 6.2 nn.TransformerEncoderLayer / TransformerEncoder 编码器

### 层定义
单层编码器由：多头自注意力 + FFN前馈网络 + 两层残差 + LayerNorm构成；Encoder用于堆叠多层编码器。
### TransformerEncoderLayer核心入参（默认值）
- d_model：特征维度【必填】
- nhead：注意力头数【必填】
- dim_feedforward：FFN隐藏维度，默认=2048
- dropout：dropout概率，默认=0.1
- activation：激活函数relu/gelu，默认='relu'
- layer_norm_eps：LN极小值，默认=1e-5
- batch_first：batch优先，默认=False
- norm_first：Pre-LN结构，默认=False
- bias：线性层启用偏置，默认=True
### TransformerEncoder参数
- encoder_layer：单层EncoderLayer实例【必填】
- num_layers：堆叠层数【必填】
- norm：顶层归一化层，默认=None
### 作用
编码输入序列全局上下文，BERT、ViT仅使用编码器结构。
### 适用场景
分类、预训练编码器模型、图像视觉Transformer

In [ ]:
# ====================== 6.2 TransformerEncoder 验证 ======================
# 创建单层编码器层：d_model=512, 8头注意力, FFN维度2048, GELU激活
enc_layer = nn.TransformerEncoderLayer(
    d_model=512, nhead=8, dim_feedforward=2048,
    activation='gelu', batch_first=True
)
# 堆叠6层编码器
encoder = nn.TransformerEncoder(enc_layer, num_layers=6)
# 生成输入序列
x_enc = torch.randn(4, 32, 512)
# 前向传播
enc_out = encoder(x_enc)
print(f"TransformerEncoder input shape: {x_enc.shape} -> output shape: {enc_out.shape}")
print(f"编码器由6层组成，每层包含多头自注意力+FFN+残差+LayerNorm")

## 6.3 nn.TransformerDecoderLayer / TransformerDecoder 解码器

### 层定义
单层解码器三层结构：掩码自注意力（屏蔽未来token）+ 交叉注意力（读取编码器输出）+ FFN。
### TransformerDecoderLayer核心入参
参数列表、默认值与TransformerEncoderLayer完全一致
### TransformerDecoder参数
- decoder_layer：单层DecoderLayer实例【必填】
- num_layers：堆叠层数【必填】
- norm：顶层归一化，默认=None
### 作用
自回归生成序列，GPT、机器翻译解码器专用。
### 适用场景
文本生成、机器翻译、图像描述生成

In [ ]:
# ====================== 6.3 TransformerDecoder 验证 ======================
# 创建单层解码器层：配置与编码器相同
dec_layer = nn.TransformerDecoderLayer(
    d_model=512, nhead=8, dim_feedforward=2048,
    activation='gelu', batch_first=True
)
# 堆叠6层解码器
decoder = nn.TransformerDecoder(dec_layer, num_layers=6)
# 目标序列（decoder输入）：batch=4, seq_len=20
x_dec = torch.randn(4, 20, 512)
# 编码器输出（memory）：作为交叉注意力的K和V
x_enc_for_dec = torch.randn(4, 32, 512)
# 前向传播：tgt=目标序列, memory=编码器输出
dec_out = decoder(x_dec, x_enc_for_dec)
print(f"TransformerDecoder input (tgt) shape: {x_dec.shape}")
print(f"TransformerDecoder memory (src) shape: {x_enc_for_dec.shape}")
print(f"TransformerDecoder output shape: {dec_out.shape}")
print("解码器比编码器多了一层交叉注意力，用于融合编码器信息")

## 6.4 nn.Transformer 完整编解码容器

### 层定义
封装多层Encoder+Decoder的完整序列翻译架构，输入源序列与目标序列完成跨序列生成。
### 核心入参（含默认值）
- d_model：特征维度【必填】
- nhead：注意力头数【必填】
- num_encoder_layers：编码器层数，默认=6
- num_decoder_layers：解码器层数，默认=6
- dim_feedforward：FFN维度，默认=2048
- dropout：全局dropout，默认=0.1
- activation：激活函数，默认='relu'
- layer_norm_eps：LN eps，默认=1e-5
- batch_first：batch优先，默认=False
- norm_first：Pre-LN，默认=False
- bias：线性层偏置，默认=True
### 适用场景
机器翻译、双语序列生成任务

In [ ]:
# ====================== 6.4 完整 Transformer 验证 ======================
# 创建完整Transformer：6层编码器+6层解码器
transformer = nn.Transformer(
    d_model=512, nhead=8,
    num_encoder_layers=6, num_decoder_layers=6,
    dim_feedforward=2048, activation='gelu', batch_first=True
)
# 源序列（编码器输入）：batch=4, seq_len=32
x_trans = torch.randn(4, 32, 512)
# 目标序列（解码器输入）：batch=4, seq_len=20
tgt_trans = torch.randn(4, 20, 512)
# 前向传播：完整编解码
trans_out = transformer(x_trans, tgt_trans)
print(f"Full Transformer src shape: {x_trans.shape}")
print(f"Full Transformer tgt shape: {tgt_trans.shape}")
print(f"Full Transformer output shape: {trans_out.shape}")
print("完整Transformer = 编码器(源序列) + 解码器(目标序列)，用于序列到序列生成任务")

# 七、网络权重初始化

## 模块说明
不属于网络层，但所有有参数层都需要正确初始化权重，控制训练初始分布，避免梯度爆炸/消失。
### 主流初始化策略（函数默认参数）
1. Kaiming(He)初始化
nn.init.kaiming_normal_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')
nn.init.kaiming_uniform_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')
2. Xavier(Glorot)初始化
nn.init.xavier_normal_(tensor, gain=1.0)
nn.init.xavier_uniform_(tensor, gain=1.0)
3. Constant常数初始化
nn.init.constant_(tensor, val)【无默认值，必须指定val】
### 使用方法
定义init_weights回调函数，模型实例调用model.apply()自动遍历所有子层完成初始化。

In [ ]:
# ====================== 7. 通用权重初始化函数 + 测试网络 ======================
# 定义初始化函数：根据层类型自动选择初始化策略
def init_weights(m):
    # 对卷积层使用Kaiming初始化（适用于ReLU系列激活函数）
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    # 对线性层使用Xavier初始化（适用于tanh/sigmoid激活函数）
    elif isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
    # 对BatchNorm层：γ初始化为1，β初始化为0
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)

# 定义测试网络：包含Conv2d、BatchNorm2d、Linear
class DemoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(16)
        self.fc = nn.Linear(16*32*32, 10)
        # 调用apply自动遍历所有子模块执行init_weights
        self.apply(init_weights)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = x.flatten(1)
        return self.fc(x)

# 实例化模型，自动完成权重初始化
model_init = DemoNet()
print("Model weight init finished")

# 测试各种初始化方法的数值分布
w_test = torch.empty(10, 10)
# Kaiming均匀分布：适用于ReLU网络
nn.init.kaiming_uniform_(w_test, a=0, mode='fan_in', nonlinearity='leaky_relu')
print(f"Kaiming uniform weight mean={w_test.mean().item():.4f}, std={w_test.std().item():.4f}")
# Xavier均匀分布：适用于tanh/sigmoid网络
nn.init.xavier_uniform_(w_test, gain=1.0)
print(f"Xavier uniform weight mean={w_test.mean().item():.4f}, std={w_test.std().item():.4f}")
print("正确的权重初始化能显著加速收敛，防止梯度消失/爆炸")

# 八、有参数层整体总结

## 分类总表
| 大类 | 代表层 | 核心可训练参数 | 典型使用场景 |
|------|--------|----------------|--------------|
| 卷积层 | Conv2d、ConvTranspose2d、DepthSepConv | 卷积核权重+偏置 | 图像特征提取、分割、检测 |
| 线性层 | Linear、Bilinear | 权重矩阵+偏置 | 分类头、多模态融合 |
| 归一化层 | BN/LN/GN | γ缩放、β偏移 | 加速训练、稳定分布 |
| 嵌入层 | Embedding、EmbeddingBag | 词向量查找表 | NLP离散编码 |
| 循环层 | LSTM/GRU/RNN | 输入/隐态权重、门控偏置 | 时序、文本序列建模 |
| Transformer | MHA、Encoder/Decoder | 注意力权重、FFN权重 | 大语言模型、视觉Transformer |

## 下一期预告
第3部分：**无参数层完整教程**（激活函数、池化、Dropout、Reshape算子等无训练参数模块）